In [1]:
cd ..

/workspace/llm-graph-construction


/opt/conda/lib/python3.10/site-packages/IPython/core/magics/osm.py:417: UserWarning: using dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [2]:
# prepare dataset

In [3]:
!conda install -y cudatoolkit
!pip install -q bitsandbytes==0.32.0
!pip install -q unsloth
!pip uninstall -q -y transformers 
!pip uninstall -q -y accelerate
!pip install -q accelerate==0.11.0
!pip install -q transformers==4.20.1
!pip install pyg_lib==0.4.0+pt22cu121 torch_scatter==2.1.2+pt22cu121 torch_sparse==0.6.18+pt22cu121 torch_cluster==1.6.3+pt22cu121 torch_spline_conv==1.2.2+pt22cu121 -f https://data.pyg.org/whl/torch-2.2.0+cu121.html
!pip install -q torch_geometric==2.4.0
!pip install torch==2.2.0 torchvision==0.17.0 torchaudio==2.2.0 --index-url https://download.pytorch.org/whl/cu121
!pip uninstall evaluate
!pip install evaluate==0.4.0

Retrieving notices: ...working... done
done
doneing environment: | 


==> WARNING: A newer version of conda exists. <==
  current version: 23.9.0
  latest version: 24.11.3

Please update conda by running

    $ conda update -n base -c defaults conda

Or to minimize the number of packages updated during conda update use

     conda install conda=24.11.3



## Package Plan ##

  environment location: /opt/conda

  added / updated specs:
    - cudatoolkit


The following packages will be downloaded:

    package                    |            build
    ---------------------------|-----------------
    ca-certificates-2024.12.31 |       h06a4308_0         128 KB
    certifi-2024.12.14         |  py310h06a4308_0         160 KB
    cudatoolkit-11.8.0         |       h6a678d5_0       630.7 MB
    openssl-3.0.15             |       h5eee18b_0         5.2 MB
    ------------------------------------------------------------
                                           Total:       636.2 MB

The fo

In [2]:
!pip uninstall -y accelerate
!pip install accelerate

Found existing installation: accelerate 0.33.0
Uninstalling accelerate-0.33.0:
  Successfully uninstalled accelerate-0.33.0
  Using cached accelerate-1.2.1-py3-none-any.whl.metadata (19 kB)
Using cached accelerate-1.2.1-py3-none-any.whl (336 kB)
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pytdc 1.1.1 requires accelerate==0.33.0, but you have accelerate 1.2.1 which is incompatible.
pytdc 1.1.1 requires datasets==2.20.0, but you have datasets 3.2.0 which is incompatible.
pytdc 1.1.1 requires evaluate==0.4.2, but you have evaluate 0.4.0 which is incompatible.
pytdc 1.1.1 requires transformers==4.43.4, but you have transformers 4.20.1 which is incompatible.
trl 0.13.0 requires transformers>=4.46.0, but you have transformers 4.20.1 which is incompatible.
unsloth 2025.1.5 requires torch>=2.4.0, but you have torch 2.2.0+cu121 which is incompatible.
unsloth 2025.1.

In [10]:
# !pip install -q unsloth
# !pip uninstall unsloth -y && pip install --upgrade --no-cache-dir "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pytdc 1.1.1 requires accelerate==0.33.0, but you have accelerate 1.2.1 which is incompatible.
pytdc 1.1.1 requires datasets==2.20.0, but you have datasets 3.2.0 which is incompatible.
pytdc 1.1.1 requires transformers==4.43.4, but you have transformers 4.48.0 which is incompatible.
lightning 2.0.8 requires pydantic<2.2.0,>=1.7.4, but you have pydantic 2.10.3 which is incompatible.
Found existing installation: unsloth 2025.1.5
Uninstalling unsloth-2025.1.5:
  Successfully uninstalled unsloth-2025.1.5
  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-oyt8amhf/unsloth_400e05fb7d8c4f71b8b7837c0410c133
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-oyt8amhf/unsloth_400e05fb7d8c4f71b8b7837c0410c133
  Resolved https://github.com/unslot

In [2]:
from training.train_and_evaluate_relation_extraction import load_stored_dataset_combination_graph
from custom_datasets.dataframe_dataset import DFDataset

/workspace/llm-graph-construction


/workspace/llm-graph-construction/graph_building/llm/OpenChat.py:10: LangChainDeprecationWarning: The class `Ollama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the :class:`~langchain-ollama package and should be used instead. To use it run `pip install -U :class:`~langchain-ollama` and import as `from :class:`~langchain_ollama import OllamaLLM``.
  ollama = Ollama(base_url='http://localhost:11434',


In [ ]:
test_name = "i2b2"

In [ ]:
dataset_train, dataset_val, dataset_test_ub = load_stored_dataset_combination_graph(balanced=True, dataset=test_name)
number_of_relations = 3

In [19]:
def construct_examples_from_df(df):
    documents = set(df["document_id"])
    instructions = []
    answers = []
    for doc in documents:
        relations_df = df[df["document_id"] == doc]
        text = ""
        relations = []
        events = set()
        for i, row in relations_df.iterrows():
            text = row["text"]
            e1 = (row["event1_start"], row["event1_end"], row["event1_text"])
            e2 = (row["event2_start"], row["event2_end"], row["event2_text"])
            relations.append((e1, row["class"], e2))
        instruction = "You are an end to end temporal relation extraction model. List event pairs from the following document, that are temporally related and should have a temporal relation computed. Answer with each event pair in one line in the format event1 -> event2.\n\n The document:\n" + text
        answer = ""
        for r in relations:
            answer += r[0][2] + "->" + r[2][2] + "\n"
        instructions.append(instruction)
        answers.append(answer)
    return instructions, answers

In [20]:
instructions, answers = construct_examples_from_df(dataset_train.df)

In [21]:
# Finetuning

In [9]:
!pip show transformers

Name: transformers
Version: 4.48.0
Summary: State-of-the-art Machine Learning for JAX, PyTorch and TensorFlow
Home-page: https://github.com/huggingface/transformers
Author: The Hugging Face team (past and future) with the help of all our contributors (https://github.com/huggingface/transformers/graphs/contributors)
Author-email: transformers@huggingface.co
License: Apache 2.0 License
Location: /opt/conda/lib/python3.10/site-packages
Requires: filelock, huggingface-hub, numpy, packaging, pyyaml, regex, requests, safetensors, tokenizers, tqdm
Required-by: flair, peft, PyTDC, transformer-smaller-training-vocab, trl, unsloth_zoo


In [1]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 2048 # Choose any! We auto support RoPE Scaling internally!
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True # Use 4bit quantization to reduce memory usage. Can be False.

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit", # or choose "unsloth/Llama-3.2-1B-Instruct"
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    # token = "hf_...", # use one if using gated models like meta-llama/Llama-2-7b-hf
)


===================================BUG REPORT===================================
Welcome to bitsandbytes. For bug reports, please submit your error trace to: https://github.com/TimDettmers/bitsandbytes/issues
For effortless bug reporting copy-paste your error into this form: https://docs.google.com/forms/d/e/1FAIpQLScPB8emS3Thkp66nvqwmjTEgxp8Y9ufuWTzFyr9kJ5AoI47dQ/viewform?usp=sf_link
CUDA_SETUP: WARNING! libcudart.so not found in any environmental path. Searching /usr/local/cuda/lib64...
CUDA SETUP: Loading binary /opt/conda/lib/python3.10/site-packages/bitsandbytes/libbitsandbytes_cpu.so...
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/opt/conda/lib/python3.10/site-packages/bitsandbytes/cuda_setup/paths.py:110: UserWarning: /usr/local/nvidia/lib:/usr/local/nvidia/lib64 did not contain libcudart.so as expected! Searching further paths...
  warn(
/opt/conda/lib/python3.10/site-packages/bitsandbytes/cextension.py:48: UserWarning: The installed version of bitsandbytes was compiled without GPU support. 8-bit optimizers and GPU quantization are unavailable.
  warn(
/opt/conda/lib/python3.10/site-packages/unsloth/__init__.py:149: UserWarning: Unsloth: Running `ldconfig /usr/lib64-nvidia` to link CUDA.
  warnings.warn(
/sbin/ldconfig.real: File /lib/x86_64-linux-gnu/libnvidia-allocator.so.1 is empty, not checked.
/sbin/ldconfig.real: File /lib/x86_64-linux-gnu/libnvidia-nscq.so.2 is empty, not checked.
/sbin/ldconfig.real: File /lib/x86_64-linux-gnu/libnvidia-allocator.so.565.57.01 is empty, not checked.
/sbin/ldconfig.real: File /lib/x86_64-linux-gnu/libnvidia-nscq.so.565.57.01 is empty, not checked.
/sbin/ldconfig.real: F

ModuleNotFoundError: No module named 'transformers.integrations.bitsandbytes'; 'transformers.integrations' is not a package